# LAB 05 — Supervised Learning: Regression

**Module:** Supervised Learning — Regression
**Session:** 05
**Duration:** 150 minutes (10-minute break included)
**Audience:** Başlangıç seviyesi Python / Machine Learning katılımcıları (LAB02, LAB03 ve LAB04'ü
tamamlamış olmalı)

## LAB02, LAB03 ve LAB04'ten Devam

Bu LAB, önceki üç oturumun doğrudan devamıdır:

- **LAB02**'de NumPy, Pandas, filtering/sorting ve groupby/aggregation öğrendiniz.
- **LAB03**'te descriptive statistics, dağılım analizi, outlier analizi ve correlation ile veriyi
  istatistiksel olarak incelediniz.
- **LAB04**'te `SimpleImputer`, `OneHotEncoder`, `StandardScaler`, train/validation/test ayrımı, data
  leakage önleme ve `Pipeline`/`ColumnTransformer` ile tekrar üretilebilir bir preprocessing workflow'u
  inşa ettiniz.

Bu LAB'da bu preprocessing bilgisini **yeni bir dataset ve yeni bir problem** üzerinde kullanacak,
üzerine gerçek regression modelleri ekleyeceğiz.

## Bu LAB'ın Ana Hedefi

Northwind sipariş verisini kullanarak, bir siparişin **taşıma ücretini (`Freight`)** tahmin eden
regression modelleri geliştirmek, bunları doğru biçimde değerlendirmek ve karşılaştırmak.

## Bu LAB'ın Sonunda Yapabilecekleriniz

1. Regression problemlerini tanımlayabileceksiniz.
2. Baseline Regression modeli oluşturabileceksiniz.
3. Linear Regression eğitebileceksiniz.
4. Multiple Linear Regression uygulayabileceksiniz.
5. Ridge ve Lasso Regularization yöntemlerini açıklayabileceksiniz.
6. Decision Tree Regression geliştirebileceksiniz.
7. MAE, MSE, RMSE ve R² hesaplayabileceksiniz.
8. Residual Analysis gerçekleştirebileceksiniz.
9. Underfitting ve Overfitting problemlerini değerlendirebileceksiniz.
10. Farklı regression modellerini aynı koşullarda karşılaştırabileceksiniz.
11. Güvenilir bir preprocessing ve model training pipeline oluşturabileceksiniz.

## Kullanılan Veri Kümesi: Northwind Orders

Bu LAB'da **Northwind Orders** veri kümesi kullanılır: `data/northwind/orders.csv`. LAB02'de
kullandığınız `customers.csv`/`orders.csv`/`shippers.csv` tablolarından yalnızca `orders.csv`'yi
kullanacağız.

### İş Problemi: Freight Charge Prediction

**Hedef değişken (target):** `Freight` — siparişin taşıma ücreti (sayısal, sürekli bir değer).

**Amaç:** Sipariş ve sevkiyat bilgilerini kullanarak, henüz bilinmeyen `Freight` tutarını tahmin eden
bir regression modeli geliştirmek.

> **Önemli bir eğitim varsayımı:** Northwind, tarihsel ve eğitim amaçlı bir veri kümesidir. `Freight`
> alanının gerçek bir kurumsal sistemde hangi aşamada (sipariş anında mı, sevkiyat sonrasında mı)
> hesaplandığı bu veri kümesinden kesin olarak çıkarılamaz. Bu LAB boyunca şu varsayımı kullanıyoruz:
> **"Model, taşıma yöntemi ve sevkiyat bilgileri belirlendikten sonra, `Freight` tutarı henüz
> bilinmiyorken tahmin yapar."** Bu varsayımın gerçek bir iş sürecinde ayrıca doğrulanması gerekir.
> Bu LAB'daki model çıktıları **gerçek taşıma fiyatlandırması veya operasyonel maliyet tahmini için
> doğrulanmış sonuçlar olarak sunulmamalıdır.**

## LAB İçeriği

| # | Bölüm | Süre |
|---|---|---|
| 1 | Regression Fundamentals & Dataset | 10 dk |
| 2 | Data Splitting & Baseline Models | 15 dk |
| 3 | Linear & Multiple Linear Regression | 20 dk |
| 4 | Ridge & Lasso Regularization | 25 dk |
| — | **Break** | 10 dk |
| 5 | Decision Tree Regression | 20 dk |
| 6 | Regression Evaluation Metrics | 20 dk |
| 7 | Residual Analysis & Model Comparison | 20 dk |
| 8 | Final Workshop & Evaluation | 10 dk |

Üç isteğe bağlı **Advanced Challenge**, ana 150 dakikalık programın dışındadır.


---
## Section 1 — Regression Fundamentals & Dataset
**Süre: 10 dakika**

### Learning Objective

Regression probleminin ne olduğunu, Classification'dan farkını ve bu LAB'ın iş problemini anlamak.

### Concept Explanation

- **What Is Regression?**: Regression, bir veya daha fazla feature kullanarak **sayısal (numerical)**
  bir hedef değişkeni tahmin etmeye çalışan bir Supervised Learning türüdür.
- **Regression vs Classification**: Classification, sonlu bir kategori kümesinden birini tahmin eder
  (örn. LAB04'teki `Reached.on.Time_Y.N`: 0 veya 1). Regression, teorik olarak sonsuz sayıda değer
  alabilen sürekli bir sayı tahmin eder (örn. bu LAB'daki `Freight`).
- **Numerical Target**: Hedef değişkenin ölçülebilir, sıralı ve aritmetik işlemlere (fark, ortalama)
  uygun bir sayısal değer olmasıdır.
- **Features and Target**: Features (`X`), tahmin için kullanılan girdi değişkenleridir. Target (`y`),
  tahmin edilmek istenen çıktı değişkenidir.
- **Supervised Learning**: Modelin, her gözlem için doğru `y` değerinin bilindiği (etiketlenmiş)
  geçmiş verilerden öğrenmesidir.
- **Regression Workflow**: Veri yükleme → feature seçimi → train/validation/test ayrımı → preprocessing
  → model eğitimi → değerlendirme → karşılaştırma.
- **Business Problem Definition**: "Freight Charge Prediction" — sipariş ve sevkiyat bilgilerine
  dayanarak taşıma ücretini tahmin etmek.

### Hands-On


In [ ]:
# Same environment setup pattern as LAB02/LAB03/LAB04: resolve the project root, add src/ to
# sys.path, then load the shared configuration and the dataset.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_northwind_orders

config = load_config()
RANDOM_STATE = config["lab05"]["random_seed"]
TARGET = config["lab05"]["target"]

df = load_northwind_orders(config)
print("Dataset shape:", df.shape)
print("Target column:", TARGET)
print("Random state (from config/settings.yaml):", RANDOM_STATE)


In [ ]:
print("Columns:", df.columns.tolist())
df.dtypes


In [ ]:
missing_summary = df.isna().sum()
print("Missing value summary (count per column):")
print(missing_summary[missing_summary > 0])


`ShippedDate` (21 eksik, henüz sevk edilmemiş siparişler), `ShipRegion` (507 eksik) ve
`ShipPostalCode` (19 eksik) sütunlarında eksik değer var. `Freight` (target) içinde **hiç eksik değer
yok** — bu, regression problemi için önemli bir ön koşuldur.

### Hands-On — Target Variable


In [ ]:
print("Freight null count:", df["Freight"].isna().sum())
print("Freight dtype      :", df["Freight"].dtype)
print("Negative Freight    :", (df["Freight"] < 0).sum())
print()
print(df["Freight"].describe())


`Freight` içinde null veya negatif değer bulunmuyor; bu, veri kümesinin bu LAB için kullanılabilir
olduğunu doğrular. Ortalama (**78.24**) medyandan (**41.36**) oldukça yüksek — bu, dağılımın **sağa
çarpık (right-skewed)** olduğuna işaret eder: çoğu sipariş düşük taşıma ücretine sahipken, az sayıda
sipariş çok yüksek bir ücrete sahip (maksimum **1007.64**).

### Hands-On — Target Distribution


In [ ]:
import matplotlib.pyplot as plt

# Color palette: fixed categorical/semantic roles reused across the ml-labs notebook series.
BAR_COLOR = "#2a78d6"
HIGHLIGHT_COLOR = "#1baf7a"
WARNING_COLOR = "#eb6834"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["Freight"], bins=40, color=BAR_COLOR)
ax.axvline(df["Freight"].mean(), color=WARNING_COLOR, linewidth=2, label=f"Mean ({df['Freight'].mean():.2f})")
ax.axvline(df["Freight"].median(), color=HIGHLIGHT_COLOR, linewidth=2, label=f"Median ({df['Freight'].median():.2f})")
ax.set_title("Freight Distribution")
ax.set_xlabel("Freight")
ax.set_ylabel("Order Count")
ax.legend(frameon=False)
style_axes(ax)
plt.show()


### Educational Question

**"Bir siparişin taşıma ücretini tahmin etmek neden bir Regression problemidir?"**

**Beklenen yanıt:** Çünkü tahmin edilmek istenen `Freight` değeri, sonlu bir kategori kümesinden
seçilmiyor — teorik olarak herhangi bir pozitif ondalıklı sayı olabilen, sürekli sayısal bir değer.

### Important

`Freight` alanını **gerçek teslimat süresi (lead time)** veya **ETA (Estimated Time of Arrival)**
olarak yorumlamayın. `Freight`, taşıma ücretiyle ilişkili parasal bir değerdir; teslimat süresi ayrı
bir konudur (bkz. `RequiredDate`/`ShippedDate`).

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: df["Freight"] için IQR yöntemiyle alt/üst sınırları hesaplayın (LAB03'teki yöntemi hatırlayın).
# TODO: Bu sınırların dışında kalan sipariş sayısını yazdırın.

# q1 = df["Freight"].quantile(0.25)
# q3 = df["Freight"].quantile(0.75)
# iqr = q3 - q1
# lower_bound = q1 - 1.5 * iqr
# upper_bound = q3 + 1.5 * iqr
# outlier_count = ((df["Freight"] < lower_bound) | (df["Freight"] > upper_bound)).sum()
# print(outlier_count)


---
## Section 2 — Data Splitting & Baseline Models
**Süre: 15 dakika**

### Learning Objective

Feature matrix ve target vector'ü doğru biçimde oluşturmak; veriyi kronolojik olarak train/validation/
test kümelerine ayırmak; baseline modellerin neden gerekli olduğunu anlamak.

### Concept Explanation

- **Feature Matrix (`X`)**: Modelin girdisi olarak kullanılan sütunların tamamı.
- **Target Vector (`y`)**: Tahmin edilmek istenen tek sütun (`Freight`).
- **Training / Validation / Test Set**: LAB04'te öğrendiğiniz ayrım burada da geçerlidir — training
  model öğrenimi için, validation model seçimi için, test ise seçilen modelin **tek seferlik**, bağımsız
  değerlendirmesi için kullanılır.
- **Baseline Regression**: Herhangi bir feature kullanmadan, basit bir istatistiksel kurala (örn. her
  zaman eğitim ortalamasını tahmin et) dayanan bir modeldir. Daha karmaşık bir modelin gerçekten faydalı
  olup olmadığını anlamak için referans noktasıdır.
- **`DummyRegressor`**: Scikit-learn'ün baseline regression için sağladığı sınıf.
- **Mean ve Median Baseline**: `DummyRegressor(strategy="mean")` her zaman training hedefinin
  ortalamasını tahmin eder; `strategy="median"` medyanı tahmin eder.
- **Reproducibility**: Aynı `random_state` ile her çalıştırıldığında aynı sonucu üreten, tekrar
  üretilebilir bir süreç.

### Feature Selection

Bu LAB'da aşağıdaki feature'lar kullanılacaktır:

| Feature | Tip | Açıklama |
|---|---|---|
| `ShipCountry` | Categorical | Sevkiyatın gönderildiği ülke |
| `ShipRegion` | Categorical | Sevkiyatın gönderildiği bölge (çoğu ülkede boş) |
| `ShipVia` | Categorical | Taşıyıcı (Shipper) kimliği — 1, 2 veya 3 |
| `Order_Month` | Numerical | `OrderDate`'ten türetilmiş sipariş ayı |
| `Order_Weekday` | Numerical | `OrderDate`'ten türetilmiş haftanın günü |
| `Planned_Lead_Time_Days` | Numerical | `RequiredDate - OrderDate` (planlanan teslim süresi, gün) |

**Dışarıda tutulan sütunlar:**

- `Freight` — target değişkenin kendisi.
- `OrderID` — bir identifier, tahmine yardımcı bir sinyal taşımaz.
- `ShippedDate` — sipariş **sevk edildikten sonra** oluşan bir bilgidir; tahmin anında bilinmesi
  beklenemez (data leakage riski).
- `CustomerID` — yüksek kardinaliteli bir identifier; varsayılan feature setine dahil edilmemiştir.
- `ShipName`, `ShipAddress`, `ShipCity`, `ShipPostalCode` — yüksek kardinaliteli adres alanları;
  gerekçesiz biçimde feature setine eklenmemiştir.
- `EmployeeID` — bu LAB'ın kapsamı dışında tutulmuştur.

> **Önemli varsayım:** `ShipVia` (taşıyıcı seçimi), siparişin **tahmin anında zaten bilindiği**
> varsayılarak feature olarak kullanılmıştır. Gerçek bir operasyonel sistemde bu varsayımın ayrıca
> doğrulanması gerekir.

### Hands-On — Feature Engineering


In [ ]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["RequiredDate"] = pd.to_datetime(df["RequiredDate"])

df["Order_Month"] = df["OrderDate"].dt.month
df["Order_Weekday"] = df["OrderDate"].dt.day_of_week
df["Planned_Lead_Time_Days"] = (df["RequiredDate"] - df["OrderDate"]).dt.days

# ShipVia is stored as an integer (1, 2, 3) but represents a carrier IDENTITY, not a quantity —
# cast it to string so it is treated as categorical, not numerical.
df["ShipVia"] = df["ShipVia"].astype(str)

print("Negative Planned_Lead_Time_Days count:", (df["Planned_Lead_Time_Days"] < 0).sum())
df[["OrderDate", "RequiredDate", "Order_Month", "Order_Weekday", "Planned_Lead_Time_Days", "ShipVia"]].head()


`Planned_Lead_Time_Days` içinde negatif değer bulunmuyor (`RequiredDate` her zaman `OrderDate`'ten
sonra) — bu, türetilen feature'ın mantıksal olarak geçerli olduğunu doğrular.

### Hands-On — Feature Matrix & Target Vector


In [ ]:
numerical_features = ["Planned_Lead_Time_Days", "Order_Month", "Order_Weekday"]
categorical_features = ["ShipCountry", "ShipRegion", "ShipVia"]
feature_columns = numerical_features + categorical_features

X = df[feature_columns]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)


### Data Splitting — Chronological Split

`OrderDate`, 1996-07-04 ile 1998-05-06 arasında güvenilir biçimde parse edilebiliyor. Bu nedenle
rastgele (random) bir split yerine **kronolojik (chronological) split** kullanacağız: veriyi tarihe
göre sıralayıp ilk %70'ini training, sonraki %15'ini validation, son %15'ini test olarak ayıracağız.

**Neden kronolojik split?** Gerçek hayatta model, geçmiş siparişlerle eğitilip **gelecekteki**
siparişler için kullanılır. Veriyi rastgele karıştırıp bölersek, model yanlışlıkla "gelecekten" bilgi
görebilir (örn. training setinde test setinden kronolojik olarak sonra gelen bir sipariş bulunabilir) —
bu, zaman serisi niteliğindeki veriler için bir leakage riskidir. Kronolojik split, gerçek kullanım
senaryosunu daha doğru simüle eder.

### Hands-On


In [ ]:
df_sorted = df.sort_values("OrderDate").reset_index(drop=True)

n = len(df_sorted)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
n_test = n - n_train - n_val

train_df = df_sorted.iloc[:n_train]
val_df = df_sorted.iloc[n_train:n_train + n_val]
test_df = df_sorted.iloc[n_train + n_val:]

print(f"Train: {len(train_df)} rows | {train_df['OrderDate'].min().date()} -> {train_df['OrderDate'].max().date()}")
print(f"Val  : {len(val_df)} rows | {val_df['OrderDate'].min().date()} -> {val_df['OrderDate'].max().date()}")
print(f"Test : {len(test_df)} rows | {test_df['OrderDate'].min().date()} -> {test_df['OrderDate'].max().date()}")


In [ ]:
X_train, y_train = train_df[feature_columns], train_df[TARGET]
X_val, y_val = val_df[feature_columns], val_df[TARGET]
X_test, y_test = test_df[feature_columns], test_df[TARGET]

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)


### Hands-On — Baseline Models


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

baseline_mean = DummyRegressor(strategy="mean").fit(X_train, y_train)
baseline_median = DummyRegressor(strategy="median").fit(X_train, y_train)

for name, model in [("DummyRegressor(mean)", baseline_mean), ("DummyRegressor(median)", baseline_median)]:
    pred_val = model.predict(X_val)
    print(
        f"{name:24s} | Val MAE: {mean_absolute_error(y_val, pred_val):.2f} | "
        f"Val RMSE: {root_mean_squared_error(y_val, pred_val):.2f} | "
        f"Val R2: {r2_score(y_val, pred_val):.3f}"
    )


İlginç bir gözlem: `median` baseline, `mean` baseline'dan daha düşük **MAE** değerine sahip (çarpık
dağılımda medyan, mutlak hataları minimize etmeye daha uygundur), ancak daha düşük (daha negatif)
**R²** değerine sahiptir (R², kareli hatayı — yani büyük hatalara karşı daha duyarlı bir ölçütü —
temel alır, ve `mean` bu ölçüt için optimal baseline'dır). Bu, **"tek bir metrik asla yeterli değildir"**
ilkesinin ilk somut örneğidir — Section 6'da bu konuya geri döneceğiz.

### Critical Rule

**Test verisini, hiçbir model seçimi veya karşılaştırması için kullanmayın.** Test seti, bu LAB'ın
sonunda, seçilmiş TEK bir model için TEK bir kez kullanılacaktır (Section 8).

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: baseline_mean modelinin TRAINING verisi üzerindeki MAE değerini hesaplayın.
# TODO: Bunu yukarıdaki validation MAE değeriyle karşılaştırın. Aralarında büyük bir fark var mı?
#       (Bir baseline modelin train/val farkının küçük olması beklenir, çünkü hiçbir feature öğrenmez.)

# pred_train_baseline = baseline_mean.predict(X_train)
# print(mean_absolute_error(y_train, pred_train_baseline))


---
## Section 3 — Linear & Multiple Linear Regression
**Süre: 20 dakika**

### Learning Objective

Simple ve Multiple Linear Regression'ın matematiğini ve Scikit-learn uygulamasını öğrenmek; model
katsayılarını (`coef_`, `intercept_`) yorumlamak.

### Mathematical Explanation

Simple Linear Regression (tek feature):

$$\hat{y}=\beta_0+\beta_1x$$

Multiple Linear Regression (birden fazla feature):

$$\hat{y}=\beta_0+\sum_{j=1}^{p}\beta_jx_j$$

- `ŷ`: Prediction (tahmin)
- `β₀`: Intercept (kesişim noktası)
- `βⱼ`: Coefficient (j'inci feature'ın katsayısı)
- `xⱼ`: Feature (j'inci girdi değişkeni)

### Part A — Simple Linear Regression (Synthetic Example)

Northwind veri kümesindeki sayısal feature'ların (`Planned_Lead_Time_Days`, `Order_Month`,
`Order_Weekday`) `Freight` ile korelasyonunu kontrol edelim.


In [ ]:
numeric_corr = train_df[numerical_features + [TARGET]].corr()[TARGET].drop(TARGET)
print("Correlation with Freight (training set):")
print(numeric_corr)


Gördüğünüz gibi, bu sayısal feature'ların hiçbiri `Freight` ile anlamlı bir doğrusal ilişkiye sahip
değil (hepsi 0'a çok yakın). Bu beklenen bir sonuçtur: `orders.csv` tablosu, taşıma ücretini belirleyen
asıl etkenleri (örn. sevkiyatın ağırlığı veya hacmi — `order_details.csv`'de bulunur, bu LAB'ın
kapsamında değil) içermiyor.

Bu nedenle, Simple Linear Regression'ın matematiğini göstermek için **küçük, açıkça etiketlenmiş bir
synthetic (yapay) örnek** kullanacağız. **Bu örnek Northwind verisinden değildir ve gerçek bir
performans sonucu olarak raporlanmamalıdır** — yalnızca eğitim amaçlıdır.

### Hands-On — [SYNTHETIC] Simple Linear Regression


In [ ]:
from sklearn.linear_model import LinearRegression

# [SYNTHETIC] Educational example only — NOT derived from the Northwind dataset.
rng = np.random.RandomState(RANDOM_STATE)
synthetic_weight = rng.uniform(1, 20, size=30)
synthetic_cost = 5 + 3 * synthetic_weight + rng.normal(0, 3, size=30)

synthetic_df = pd.DataFrame({"Package_Weight_kg": synthetic_weight, "Shipping_Cost": synthetic_cost})

simple_model = LinearRegression()
simple_model.fit(synthetic_df[["Package_Weight_kg"]], synthetic_df["Shipping_Cost"])

print("[SYNTHETIC] coef_     :", simple_model.coef_)
print("[SYNTHETIC] intercept_:", simple_model.intercept_)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(synthetic_df["Package_Weight_kg"], synthetic_df["Shipping_Cost"], color=BAR_COLOR, label="[SYNTHETIC] data points")

x_line = np.linspace(synthetic_df["Package_Weight_kg"].min(), synthetic_df["Package_Weight_kg"].max(), 100)
y_line = simple_model.predict(pd.DataFrame({"Package_Weight_kg": x_line}))
ax.plot(x_line, y_line, color=WARNING_COLOR, linewidth=2, label="Fitted line")

ax.set_title("[SYNTHETIC] Simple Linear Regression Example")
ax.set_xlabel("Package_Weight_kg")
ax.set_ylabel("Shipping_Cost")
ax.legend(frameon=False)
style_axes(ax)
plt.show()


**Yorum:** `coef_` ≈ 2.78, verideki gerçek üretim formülüne (`5 + 3 * weight + noise`) oldukça yakın —
model, `Package_Weight_kg` 1 birim arttığında `Shipping_Cost`'un yaklaşık 2.78 birim artacağını
öğrendi. `intercept_` ≈ 6.51; gerçek üretim formülündeki sabit terim 5'tir, ancak 30 gözlemlik küçük
örneklemde eklenen rastgele gürültü (`noise`) nedeniyle öğrenilen katsayı ve kesişim noktası, gerçek
değerlere **yakın ama tam eşit değildir** — bu da küçük örneklem büyüklüğünün tahmin belirsizliği
üzerindeki etkisinin iyi bir örneğidir. **Bu sonuçlar yalnızca synthetic örnek içindir, Northwind
`Freight` tahmini ile bir ilgisi yoktur.**

### Part B — Multiple Linear Regression (Real Northwind Data)

Şimdi gerçek Northwind verisine dönüyoruz. LAB04'te öğrendiğiniz preprocessing bileşenlerini
kullanacağız, ancak bu LAB'ın farklı şeması için **yeni bir preprocessing pipeline'ı** inşa edeceğiz
(LAB04'teki E-Commerce Shipping pipeline'ı bu dataset için uygun değildir).

**Preprocessing tasarım kararları:**

- **Numerical features** (`Planned_Lead_Time_Days`, `Order_Month`, `Order_Weekday`): `SimpleImputer`
  (median) + `StandardScaler`.
- **Categorical features** (`ShipCountry`, `ShipRegion`, `ShipVia`): `SimpleImputer`
  (`strategy="constant", fill_value="Unknown"`) + `OneHotEncoder(handle_unknown="ignore")`.

**Neden `constant="Unknown"` ve `most_frequent` değil?** `ShipRegion`'daki eksik değerler **rastgele
değil** — hangi ülkeye gönderildiğiyle ilişkili yapısal bir örüntü taşıyor (örn. ABD/Kanada/Brezilya
siparişlerinde `ShipRegion` hep dolu, Almanya/Fransa gibi çoğu Avrupa ülkesinde hep boş). Bu durumda
eksikliği en sık görülen değerle doldurmak (`most_frequent`) yanıltıcı olur; eksikliğin kendisini açık
bir `"Unknown"` kategorisi olarak korumak daha doğrudur.

### Hands-On


In [ ]:
country_region_missing = df.groupby("ShipCountry")["ShipRegion"].apply(lambda s: s.isna().mean()).sort_values()
print("Share of missing ShipRegion values, by ShipCountry:")
print(country_region_missing)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)


### Critical Engineering Rule

Preprocessing ve estimator (model), **aynı `Pipeline` içinde** bulunmalıdır. Bu hem leakage'ı önler
(pipeline yalnızca `.fit()` çağrıldığı veride öğrenir) hem de kodu tekrar üretilebilir kılar.


In [ ]:
linear_pipe = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LinearRegression()),
])

linear_pipe.fit(X_train, y_train)

pred_train_linear = linear_pipe.predict(X_train)
pred_val_linear = linear_pipe.predict(X_val)

print(f"Train MAE: {mean_absolute_error(y_train, pred_train_linear):.2f}")
print(f"Val MAE  : {mean_absolute_error(y_val, pred_val_linear):.2f}")
print(f"Val RMSE : {root_mean_squared_error(y_val, pred_val_linear):.2f}")
print(f"Val R2   : {r2_score(y_val, pred_val_linear):.3f}")


Baseline'a (Val MAE ≈ 69.85, Val R² ≈ 0.000) kıyasla, Multiple Linear Regression hem MAE'yi (≈ 63.78)
düşürdü hem de R²'yi (≈ 0.147) pozitif yaptı — model, feature'lardan gerçek bir sinyal öğrendi, ancak
açıklanan varyans oranı (R² ≈ %15) hâlâ düşük: `Freight`'ı belirleyen asıl etkenlerin (ağırlık, hacim
gibi) bu feature setinde bulunmadığını gösteriyor.

### Part C — Model Interpretation


In [ ]:
feature_names = linear_pipe.named_steps["preprocessing"].get_feature_names_out()
coefficients = pd.Series(linear_pipe.named_steps["model"].coef_, index=feature_names)

print("Intercept:", linear_pipe.named_steps["model"].intercept_)
print()
print("Top 10 coefficients by absolute magnitude:")
print(coefficients.reindex(coefficients.abs().sort_values(ascending=False).index).head(10))


En büyük katsayılar `ShipCountry` ve `ShipRegion` dummy değişkenlerine ait (örn. `ShipCountry_Austria`
≈ +108.65, `ShipRegion_ID` ≈ +92.90) — bu, varış ülkesinin `Freight` üzerinde sayısal feature'lardan
(ay, haftanın günü, planlanan teslim süresi) çok daha güçlü bir etkiye sahip olduğunu gösteriyor
(bu, LAB03 tarzı bir `groupby("ShipCountry")["Freight"].mean()` analizi ile de doğrulanabilir).

### Important

- Bu katsayılar **nedensel (causal)** bir etki iddiası değildir — yalnızca bu veri kümesindeki
  **gözlemsel ilişkiyi** yansıtır.
- `StandardScaler` uygulanan sayısal feature'ların katsayıları, **ölçeklenmiş** birimler üzerinden
  yorumlanmalıdır — doğrudan orijinal birimlerle (örn. "gün") karıştırılmamalıdır. Kategorik dummy
  değişkenlerin katsayıları ise referans kategoriye göre fark olarak yorumlanır.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: coefficients Series'inde en küçük (sıfıra en yakın) 5 katsayıyı yazdırın.
# TODO: Bunların hangi feature'lara ait olduğunu yorumlayın — Freight üzerinde en az etkiye sahip
#       görünen feature'lar hangileri?

# print(coefficients.reindex(coefficients.abs().sort_values(ascending=True).index).head(5))


---
## Section 4 — Ridge & Lasso Regularization
**Süre: 25 dakika**

### Learning Objective

Regularization'ın ne olduğunu, Ridge (L2) ve Lasso (L1) arasındaki farkı ve `alpha` hiperparametresinin
etkisini anlamak.

### Concept Explanation

- **Why Regularization?**: Çok sayıda feature (özellikle one-hot encoded kategorik değişkenler) model
  karmaşıklığını artırır ve overfitting riskini yükseltir. Regularization, katsayıların büyüklüğüne
  bir ceza (penalty) ekleyerek modeli basitleştirir.
- **L2 Regularization (Ridge)**: Katsayıların karelerinin toplamına ceza uygular; katsayıları **sıfıra
  yaklaştırır** ama genellikle tam sıfıra indirmez.
- **L1 Regularization (Lasso)**: Katsayıların mutlak değerlerinin toplamına ceza uygular; bazı
  katsayıları **tam sıfıra** indirebilir — bu da örtük bir feature selection etkisi yaratabilir.
- **Alpha Hyperparameter**: Ceza teriminin gücünü kontrol eder. `alpha=0` düzenlileştirme yapmaz (düz
  Linear Regression'a yaklaşır); `alpha` büyüdükçe katsayılar daha fazla küçülür/sıfırlanır.
- **Scaling and Regularization**: Regularization, feature'ların ölçeğine duyarlıdır — bu yüzden
  `StandardScaler` (preprocessing pipeline'ımızda zaten mevcut) burada özellikle önemlidir.

### Hands-On — Ridge


In [ ]:
from sklearn.linear_model import Ridge

ridge_pipe = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
])
ridge_pipe.fit(X_train, y_train)

pred_val_ridge = ridge_pipe.predict(X_val)
print(f"Ridge(alpha=1.0) | Val MAE: {mean_absolute_error(y_val, pred_val_ridge):.2f} | "
      f"Val RMSE: {root_mean_squared_error(y_val, pred_val_ridge):.2f} | "
      f"Val R2: {r2_score(y_val, pred_val_ridge):.3f}")


### Hands-On — Lasso at Different Alpha Values


In [ ]:
import warnings
from sklearn.linear_model import Lasso

lasso_results = []
lasso_models = {}

for alpha in [0.01, 0.1, 1.0, 10.0]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pipe = Pipeline(steps=[("preprocessing", preprocessor), ("model", Lasso(alpha=alpha, random_state=RANDOM_STATE))])
        pipe.fit(X_train, y_train)
        if caught:
            print(f"alpha={alpha}: {len(caught)} warning(s) raised -> {[str(w.message) for w in caught]}")

    pred_val_lasso = pipe.predict(X_val)
    lasso_models[alpha] = pipe
    lasso_results.append({
        "alpha": alpha,
        "Val_MAE": mean_absolute_error(y_val, pred_val_lasso),
        "Val_RMSE": root_mean_squared_error(y_val, pred_val_lasso),
        "Val_R2": r2_score(y_val, pred_val_lasso),
        "n_iter": pipe.named_steps["model"].n_iter_,
    })

pd.DataFrame(lasso_results)


Bu çalıştırmada **hiçbir convergence warning oluşmadı** (`n_iter` değerleri, `Lasso`'nun varsayılan
`max_iter=1000` sınırının çok altında kaldı). Eğer bir convergence warning görseniz, bunu yalnızca
gizlemek (örn. `warnings.filterwarnings("ignore")`) yerine, genellikle `max_iter` değerini artırmak
veya feature'ların doğru ölçeklendiğini (`StandardScaler`) kontrol etmek doğru çözümdür.

`alpha` arttıkça (0.01 → 10.0) Val R² **0.147'den 0.000'a** kadar düşüyor — `alpha=10.0`'da model,
tamamen baseline'a (`DummyRegressor(mean)`) eşdeğer hale geliyor.

### Hands-On — Coefficient Shrinkage Comparison


In [ ]:
ridge_coefs = ridge_pipe.named_steps["model"].coef_
lasso_1_coefs = lasso_models[1.0].named_steps["model"].coef_
lasso_10_coefs = lasso_models[10.0].named_steps["model"].coef_

print(f"Ridge(alpha=1.0)  exact-zero coefficients: {(ridge_coefs == 0).sum()} / {len(ridge_coefs)}")
print(f"Lasso(alpha=1.0)  exact-zero coefficients: {(lasso_1_coefs == 0).sum()} / {len(lasso_1_coefs)}")
print(f"Lasso(alpha=10.0) exact-zero coefficients: {(lasso_10_coefs == 0).sum()} / {len(lasso_10_coefs)}")
print()
print(f"Lasso(alpha=10.0) intercept: {lasso_models[10.0].named_steps['model'].intercept_:.2f}")
print(f"Training set Freight mean  : {y_train.mean():.2f}")


### Critical Concepts

- **Ridge, bu veri kümesinde hiçbir katsayıyı tam sıfıra indirmedi** (47 katsayının 0'ı tam sıfır) —
  Ridge, katsayıları küçültür ama nadiren tam olarak eler.
- **Lasso, `alpha=1.0`'da 47 katsayıdan 37'sini tam sıfıra indirdi**, `alpha=10.0`'da ise **47
  katsayının tamamını** sıfırladı — model artık hiçbir feature kullanmıyor, tahmini `intercept_`
  (≈ 77.02) training setinin `Freight` ortalamasına (≈ 77.02 — Section 1'deki 78.24 değeri **tüm**
  veri kümesinin ortalamasıdır, training alt kümesinin ortalaması biraz farklıdır) tam olarak eşit,
  yani model fiilen baseline'a dönüştü.
- Bu davranış **bu özel dataset ve bu alpha değerleri için gözlemlenmiştir** — Lasso'nun her dataset
  üzerinde mutlaka feature selection yapacağını veya her `alpha`'da aynı oranda sıfırlama
  göstereceğini iddia etmiyoruz; bu, alpha büyüklüğüne ve feature'ların gerçek sinyal gücüne bağlıdır.

### Technical Note

Ridge objective function: $\min_\beta \|y - X\beta\|^2 + \alpha\|\beta\|_2^2$

Lasso objective function: $\min_\beta \|y - X\beta\|^2 + \alpha\|\beta\|_1$

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Yukarıdaki lasso_results tablosundan, Val_RMSE değeri en düşük olan alpha değerini bulun.
# TODO: Bu alpha değerinin Ridge(alpha=1.0) performansıyla nasıl karşılaştığını yorumlayın.

# best_lasso_row = pd.DataFrame(lasso_results).sort_values("Val_RMSE").iloc[0]
# print(best_lasso_row)


---
## ☕ Break — 10 dakika

Kısa bir ara verelim. Devam ederken: preprocessing pipeline'ımız (`preprocessor`), `X_train`/`X_val`/
`X_test` değişkenleri ve `TARGET`/`numerical_features`/`categorical_features` tanımları bundan sonraki
tüm bölümlerde kullanılacaktır.


---
## Section 5 — Decision Tree Regression
**Süre: 20 dakika**

### Learning Objective

Decision Tree Regression'ın temel mantığını öğrenmek ve `max_depth` ile model karmaşıklığının
underfitting/overfitting üzerindeki etkisini gözlemlemek.

### Concept Explanation

- **Decision Tree Fundamentals**: Bir Decision Tree, veriyi feature değerlerine göre art arda ikiye
  bölerek (splitting) bir ağaç yapısı oluşturur.
- **Regression Tree**: Her **leaf node** (yaprak düğüm), o yaprağa düşen training örneklerinin hedef
  değerlerinin **ortalamasını** tahmin olarak döndürür.
- **Splitting**: Her düğümde, hatayı (örn. kareli hata) en çok azaltan feature ve eşik değeri seçilir.
- **Tree Depth (`max_depth`)**: Ağacın kök düğümden bir yaprağa kadar olan maksimum katman sayısı.
  Derinlik arttıkça model daha karmaşık, daha "detaycı" hale gelir.
- **Minimum Samples Split / Leaf**: Bir düğümün bölünebilmesi veya bir yaprağın oluşabilmesi için
  gereken minimum örnek sayısı — bunlar da karmaşıklığı sınırlayan parametrelerdir.
- **Overfitting**: Model, training verisindeki gürültüyü (noise) bile ezberlediğinde; training
  hatası çok düşük ama validation hatası yüksek olur.
- **Tree-Based vs Linear Models**: Decision Tree'ler, feature'lar arasındaki doğrusal olmayan
  ilişkileri ve etkileşimleri yakalayabilir, ancak linear model'lerin aksine ölçeklendirmeye
  (scaling) genellikle ihtiyaç duymaz.

### Hands-On


In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_results = []
tree_models = {}

for depth in [2, 4, 6, None]:
    pipe = Pipeline(steps=[("preprocessing", preprocessor), ("model", DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    tree_models[depth] = pipe

    pred_train_tree = pipe.predict(X_train)
    pred_val_tree = pipe.predict(X_val)
    tree_results.append({
        "max_depth": depth,
        "Train_MAE": mean_absolute_error(y_train, pred_train_tree),
        "Val_MAE": mean_absolute_error(y_val, pred_val_tree),
        "Val_R2": r2_score(y_val, pred_val_tree),
    })

tree_results_df = pd.DataFrame(tree_results)
tree_results_df


### Interpretation

`max_depth` arttıkça bir **underfitting -> good fit -> overfitting** geçişi gözlemliyoruz:

- `max_depth=2`: Train MAE ve Val MAE birbirine yakın (≈ 65.5 vs ≈ 64.6) — basit ama tutarlı bir model.
- `max_depth=4`: Train MAE düşüyor (≈ 61.6) ama Val R² **negatife** dönüyor — model validation setinde
  baseline'dan daha kötü hale geliyor.
- `max_depth=6`: Train/Val farkı daha da büyüyor.
- `max_depth=None` (sınırsız derinlik): Train MAE çok düşük (≈ 5.8 — model training verisini neredeyse
  ezberledi) ama Val MAE çok yüksek (≈ 88.8) ve Val R² ciddi şekilde negatif (≈ -1.48) — bu, **klasik
  bir overfitting örneğidir.**

### Hands-On — Visualizing a Small Tree


In [ ]:
from sklearn.tree import plot_tree

small_tree_model = tree_models[2].named_steps["model"]
small_tree_feature_names = tree_models[2].named_steps["preprocessing"].get_feature_names_out()

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(small_tree_model, feature_names=small_tree_feature_names, filled=True, fontsize=8, ax=ax)
plt.show()


Not: Yalnızca `max_depth=2` olan küçük ağacı görselleştirdik — `max_depth=None` gibi çok büyük bir
ağacı çizdirmek okunaksız olur ve bu LAB'ın kapsamı dışındadır.

### Important

Decision Tree'ler, Linear/Ridge/Lasso'nun aksine feature scaling'e ihtiyaç duymaz (bölme kararları,
feature'ın mutlak ölçeğinden değil sıralamasından etkilenir). Ancak burada **adil bir karşılaştırma**
için tüm modellerde (Linear, Ridge, Lasso, Decision Tree) **aynı `preprocessor`** (dolayısıyla aynı
ölçeklenmiş/encoded feature'lar) kullanıldı — bu, Decision Tree için gerekli olmasa da sonuçları
yanlış yönde etkilemez ve karşılaştırmayı tekrar üretilebilir kılar. Bu tercih burada açıkça
belgelenmiştir.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: tree_results_df tablosunda, Train_MAE ile Val_MAE arasındaki farkı (Val_MAE - Train_MAE) yeni
#       bir sütun olarak ekleyin.
# TODE: Bu farkın max_depth ile nasıl değiştiğini yorumlayın (büyüyen fark = artan overfitting sinyali).

# tree_results_df["Train_Val_Gap"] = tree_results_df["Val_MAE"] - tree_results_df["Train_MAE"]
# print(tree_results_df)


---
## Section 6 — Regression Evaluation Metrics
**Süre: 20 dakika**

### Learning Objective

MAE, MSE, RMSE ve R²'yi matematiksel olarak ve Scikit-learn ile hesaplamak; bu metriklerin ne zaman
hangi davranışı gösterdiğini anlamak.

### Mathematical Formulas

Mean Absolute Error:

$$MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

Mean Squared Error:

$$MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2$$

Root Mean Squared Error:

$$RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

R-Squared:

$$R^2=1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}$$

### Hands-On — Manual Formula Check

Scikit-learn'ün `mean_absolute_error` fonksiyonunun, yukarıdaki formülle tutarlı olduğunu
`linear_pipe`'ın validation tahminleri üzerinde doğrulayalım.


In [ ]:
manual_mae = np.mean(np.abs(y_val.values - pred_val_linear))
sklearn_mae = mean_absolute_error(y_val, pred_val_linear)

print("Manual MAE  :", round(manual_mae, 4))
print("Sklearn MAE :", round(sklearn_mae, 4))
print("Match:", np.isclose(manual_mae, sklearn_mae))


### Hands-On — Model Evaluation Table

Bu LAB boyunca eğittiğimiz tüm modelleri, aynı training/validation bölmesi üzerinde karşılaştıralım.


In [ ]:
def evaluate_pipe(name, pipe):
    pred_train = pipe.predict(X_train)
    pred_val = pipe.predict(X_val)
    return {
        "Model": name,
        "Train_MAE": mean_absolute_error(y_train, pred_train),
        "Validation_MAE": mean_absolute_error(y_val, pred_val),
        "Validation_MSE": mean_squared_error(y_val, pred_val),
        "Validation_RMSE": root_mean_squared_error(y_val, pred_val),
        "Validation_R2": r2_score(y_val, pred_val),
    }


from sklearn.metrics import mean_squared_error

evaluation_rows = [
    {
        "Model": "DummyRegressor(mean)",
        "Train_MAE": mean_absolute_error(y_train, baseline_mean.predict(X_train)),
        "Validation_MAE": mean_absolute_error(y_val, baseline_mean.predict(X_val)),
        "Validation_MSE": mean_squared_error(y_val, baseline_mean.predict(X_val)),
        "Validation_RMSE": root_mean_squared_error(y_val, baseline_mean.predict(X_val)),
        "Validation_R2": r2_score(y_val, baseline_mean.predict(X_val)),
    },
    evaluate_pipe("LinearRegression", linear_pipe),
    evaluate_pipe("Ridge(alpha=1.0)", ridge_pipe),
    evaluate_pipe("Lasso(alpha=0.1)", lasso_models[0.1]),
    evaluate_pipe("Lasso(alpha=1.0)", lasso_models[1.0]),
    evaluate_pipe("DecisionTree(max_depth=2)", tree_models[2]),
    evaluate_pipe("DecisionTree(max_depth=None)", tree_models[None]),
]

evaluation_df = pd.DataFrame(evaluation_rows).round(3)
evaluation_df


### Technical Interpretation

- **MAE**, hedefle aynı birimdedir (`Freight` para birimi) ve yorumlaması en sezgisel metriktir.
- **MSE**, hataların karesini aldığı için büyük hatalara (outlier'lara) orantısız bir ağırlık verir.
- **RMSE**, `MSE`'nin karekökü olduğu için yine hedefle aynı birimdedir, ama büyük hatalara MAE'den
  daha duyarlıdır.
- **R²**, modelin açıkladığı varyans oranıdır; **negatif olabilir** (bu, modelin basit bir ortalama
  tahmininden daha kötü performans gösterdiği anlamına gelir — bu tabloda `DecisionTree(max_depth=None)`
  için gözlemlendi).
- Tek bir metrik, bir modelin bütün davranışını açıklamaz — Section 2'de gördüğümüz mean/median
  baseline karşılaştırması bunun açık bir örneğidir.

### Business Interpretation

"`MAE = 10`" sonucunu tek başına "başarılı" veya "başarısız" olarak değerlendirmeyin. `Freight`'ın
ölçeğini (ortalama ≈ 78, std ≈ 117) ve iş beklentilerini (örn. "10 birimlik bir hata operasyonel
olarak kabul edilebilir mi?") dikkate almadan bir MAE değeri yorumlanamaz.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: evaluation_df tablosunu Validation_RMSE sütununa göre artan sırada sıralayın.
# TODO: En iyi (en düşük RMSE) ve en kötü (en yüksek RMSE) modeli yazdırın.

# sorted_df = evaluation_df.sort_values("Validation_RMSE")
# print(sorted_df[["Model", "Validation_RMSE"]])


---
## Section 7 — Residual Analysis & Model Comparison
**Süre: 20 dakika**

### Learning Objective

Residual (hata) analiziyle bir modelin hangi koşullarda iyi/kötü performans gösterdiğini incelemek;
modelleri yalnızca tek bir metrikle değil, çok yönlü kriterlerle karşılaştırmak.

### Residual Definition

$$e_i=y_i-\hat{y}_i$$

### Concept Explanation

- **Residual**: Gerçek değer ile tahmin arasındaki fark.
- **Heteroscedasticity**: Hata büyüklüğünün, tahmin edilen (veya gerçek) değere göre sistematik olarak
  değişmesi (örn. büyük `Freight` değerlerinde hatanın da büyümesi).
- **Underfitting / Overfitting**: Section 5'te gördüğümüz gibi, train/validation hata farkı bu
  problemlerin bir göstergesidir.
- **Generalization**: Modelin, görmediği (unseen) veride de tutarlı performans göstermesi.

### Hands-On — Residual Plots (LinearRegression, Validation Set)


In [ ]:
residuals_linear = y_val.values - pred_val_linear

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_val, pred_val_linear, color=BAR_COLOR, alpha=0.7)
min_v, max_v = y_val.min(), y_val.max()
axes[0].plot([min_v, max_v], [min_v, max_v], color=WARNING_COLOR, linewidth=2, linestyle="--")
axes[0].set_title("Actual vs Predicted")
axes[0].set_xlabel("Actual Freight")
axes[0].set_ylabel("Predicted Freight")
style_axes(axes[0])

axes[1].scatter(pred_val_linear, residuals_linear, color=BAR_COLOR, alpha=0.7)
axes[1].axhline(0, color=WARNING_COLOR, linewidth=2, linestyle="--")
axes[1].set_title("Residuals vs Predicted")
axes[1].set_xlabel("Predicted Freight")
axes[1].set_ylabel("Residual")
style_axes(axes[1])

axes[2].hist(residuals_linear, bins=25, color=BAR_COLOR)
axes[2].axvline(0, color=WARNING_COLOR, linewidth=2, linestyle="--")
axes[2].set_title("Residual Distribution")
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Count")
style_axes(axes[2])

plt.tight_layout()
plt.show()


In [ ]:
print(pd.Series(residuals_linear).describe())


### Interpretation

- Residual'ların ortalaması (≈ +5.3) sıfıra yakın ama tam sıfır değil — model hafifçe hedefi
  **underestimate** etme eğiliminde.
- "Residuals vs Predicted" grafiğinde, tahmin değeri arttıkça hata dağılımının genişlediği görülüyor
  — bu bir **heteroscedasticity** belirtisidir: model, düşük `Freight` değerlerinde daha tutarlı,
  yüksek `Freight` değerlerinde daha belirsiz.
- Residual std'si (≈ 102.6) oldukça yüksek ve dağılımda tek bir çok büyük pozitif outlier var
  (maksimum residual ≈ +504) — bu, modelin **olağanüstü yüksek bir `Freight` değerini çok düşük
  tahmin ettiği** bir siparişe ait.

### Hands-On — Investigating the Largest Error


In [ ]:
val_with_predictions = val_df.copy()
val_with_predictions["Predicted_Freight"] = pred_val_linear
val_with_predictions["Residual"] = residuals_linear

largest_error_row = val_with_predictions.loc[val_with_predictions["Residual"].abs().idxmax()]
largest_error_row[["OrderDate", "ShipCountry", "ShipRegion", "ShipVia", "Freight", "Predicted_Freight", "Residual"]]


Bu sipariş (İrlanda'ya gönderilen, gerçek `Freight` ≈ 603.5) model tarafından ≈ 99.4 olarak tahmin
edilmiş — bu ülke için training verisinde yeterince yüksek-`Freight` örnek olmayabilir, ya da bu
sipariş gerçekten istisnai (örn. çok büyük bir sevkiyat). Tek bir feature setiyle (ülke, ay, haftanın
günü, planlanan süre) bu tür istisnai siparişleri yakalamak zordur.

### Optional Group Analysis — Error by ShipCountry


In [ ]:
country_mae = (
    val_with_predictions
    .groupby("ShipCountry")
    .apply(lambda d: pd.Series({"MAE": mean_absolute_error(d["Freight"], d["Predicted_Freight"]), "n": len(d)}))
)
country_mae = country_mae[country_mae["n"] >= 5].sort_values("MAE", ascending=False)
country_mae


**Önemli sınırlama:** Bu gruplardaki örnek sayıları çok küçük (5-16 sipariş). Bu tablo, "model ABD
siparişlerinde kötü" gibi kesin bir sonuç çıkarmak için **yeterli istatistiksel güce sahip değildir**
— yalnızca ileri incelemeye değer bir ön gözlem olarak ele alınmalıdır.

### Model Comparison

Modelleri karşılaştırırken dikkate alınması gereken kriterler:

- **Predictive Performance**: Section 6'daki metrik tablosu.
- **Generalization**: Train/Validation farkı (Section 5).
- **Complexity**: Kaç hiperparametre, kaç feature kullanılıyor.
- **Explainability**: Linear modellerin `coef_`'leri, Decision Tree'nin düşük derinlikte görsel
  yorumlanabilirliği.
- **Training Cost**: Bu küçük veri kümesinde ihmal edilebilir, ancak genel bir kriterdir.

### Critical Rule

**Validation verisi model seçimi için kullanılabilir** (yukarıdaki gibi). **Test verisi, Section 8'de
seçilen TEK modelin bağımsız değerlendirmesi için, bir kez kullanılmalıdır.**

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: val_with_predictions içinde Residual sütununun mutlak değerine göre en küçük 5 satırı bulun
#       (modelin en iyi tahmin ettiği siparişler).
# TODO: Bu siparişlerin ShipCountry değerlerine bakın — herhangi bir örüntü fark ediyor musunuz?

# best_predictions = val_with_predictions.reindex(val_with_predictions["Residual"].abs().sort_values().index).head(5)
# print(best_predictions[["ShipCountry", "Freight", "Predicted_Freight", "Residual"]])


---
## Section 8 — Final Workshop & Evaluation
**Süre: 10 dakika**

### Workshop Objective

Bu LAB boyunca geliştirdiğiniz regression modellerini teknik olarak karşılaştırıp, **tek bir model**
seçip, bunu test verisinde **bir kez** değerlendireceksiniz.

### Tasks

1. Baseline performansını belirleyin (Section 2 / Section 6 tablosu).
2. Linear Regression modelinin validation MAE değerini hesaplayın (Section 3).
3. Ridge veya Lasso modeliyle karşılaştırın (Section 4 / Section 6).
4. Decision Tree sonucunu değerlendirin (Section 5).
5. En uygun model adayını seçin.
6. Seçiminizi gerekçelendirin.
7. Seçilen modeli test verisinde **bir kez** değerlendirin.
8. Sonuçların sınırlamalarını açıklayın.

### Model Selection

`evaluation_df` tablosuna göre (Section 6), `LinearRegression` ve `Lasso(alpha=0.1)` validation
performansı açısından birbirine çok yakın ve mevcut adaylar arasında en iyi sonucu veriyor
(Val R² ≈ 0.14-0.15). `LinearRegression`'ı şu nedenlerle final model olarak seçiyoruz:

- Ayarlanması gereken bir `alpha` hiperparametresi yok (daha basit).
- Katsayıları doğrudan yorumlanabilir (Section 3, Part C).
- Validation performansı, regularized alternatiflere kıyasla **hiçbir dezavantaj** göstermiyor.

### Hands-On — Final Test Evaluation (ONE TIME ONLY)


In [ ]:
# This is the ONLY place in this notebook where the test set is used.
# linear_pipe was already fit exclusively on X_train/y_train in Section 3 — we do not refit here.
pred_test_final = linear_pipe.predict(X_test)

final_test_mae = mean_absolute_error(y_test, pred_test_final)
final_test_mse = mean_squared_error(y_test, pred_test_final)
final_test_rmse = root_mean_squared_error(y_test, pred_test_final)
final_test_r2 = r2_score(y_test, pred_test_final)

print(f"Test MAE : {final_test_mae:.2f}")
print(f"Test MSE : {final_test_mse:.2f}")
print(f"Test RMSE: {final_test_rmse:.2f}")
print(f"Test R2  : {final_test_r2:.3f}")


Test seti performansı (Val R² ≈ 0.147, Test R² ≈ 0.184) validation setiyle aynı büyüklük
mertebesinde — modelin validation setine aşırı uydurulmadığını (overfit edilmediğini) gösteriyor.
Test MAE (≈ 71.6), validation MAE'den (≈ 63.8) biraz daha yüksek; bu fark, test döneminin (1998
Mart-Mayıs) farklı bir sipariş karması içerebileceğiyle açıklanabilir.

### Expected Conclusion — Model Evaluation Summary

| Başlık | Değer |
|---|---|
| **Dataset** | Northwind Orders (`data/northwind/orders.csv`, 830 sipariş) |
| **Target** | `Freight` (sürekli, sayısal; eksik/negatif değer yok) |
| **Features** | `ShipCountry`, `ShipRegion`, `ShipVia` (categorical); `Planned_Lead_Time_Days`, `Order_Month`, `Order_Weekday` (numerical) |
| **Split Strategy** | Kronolojik 70/15/15 (train: 581, val: 124, test: 125 sipariş) |
| **Baseline** | `DummyRegressor(mean)` — Val MAE ≈ 69.85, Val R² ≈ 0.000 |
| **Compared Models** | DummyRegressor, LinearRegression, Ridge, Lasso (4 alpha), DecisionTreeRegressor (4 depth) |
| **Selected Model** | `LinearRegression` (preprocessing pipeline dahil) |
| **Final Test Metrics** | MAE ≈ 71.57, RMSE ≈ 123.13, R² ≈ 0.184 |
| **Error Analysis** | Heteroscedasticity (yüksek `Freight` değerlerinde daha büyük hata); tek bir aşırı outlier (İrlanda siparişi) |
| **Limitations** | Gerçek taşıma maliyetini belirleyen ağırlık/hacim bilgisi feature setinde yok; R² düşük (≈ %15-18); ülke bazlı gruplar küçük örneklemli |
| **Future Improvements** | `order_details.csv` ile ağırlık/miktar bilgisinin eklenmesi; daha fazla geçmiş veri; `shippers.csv` ile taşıyıcı bilgisinin zenginleştirilmesi |

### Important

**En düşük validation/test hatasına sahip modelin, mutlaka production için en uygun model olduğu
iddia edilmemektedir.** Model seçimi; performans, karmaşıklık, yorumlanabilirlik, eğitim maliyeti ve
iş gereksinimleri gibi birden fazla faktörün birlikte değerlendirilmesini gerektirir.


---
## Advanced Challenges (İsteğe Bağlı)

Aşağıdaki üç challenge, ana 150 dakikalık programın **dışındadır** ve isteğe bağlıdır. Deneyimli
katılımcılar için tasarlanmıştır.

### Challenge 1 — Regularization Path

Farklı `alpha` değerleri için Ridge ve Lasso katsayılarının nasıl değiştiğini (coefficient path)
görselleştirin.


In [ ]:
# Advanced Challenge 1 — Regularization Path (optional):
# TODO: np.logspace(-3, 2, 20) gibi geniş bir alpha aralığı tanımlayın.
# TODO: Her alpha için bir Ridge (veya Lasso) pipeline'ı fit edin ve coef_ değerlerini bir listeye kaydedin.
# TODO: alpha (x ekseni, log scale) ile her feature'ın katsayısını (y ekseni) çizin.
# TODO: Ridge ve Lasso path'lerini karşılaştırın — Lasso'da katsayıların tam sıfıra "iniş" şeklini
#       gözlemleyin, Ridge'de yalnızca küçülme olduğunu doğrulayın.

# alphas = np.logspace(-3, 2, 20)
# ridge_coef_path = []
# for a in alphas:
#     p = Pipeline(steps=[("preprocessing", preprocessor), ("model", Ridge(alpha=a, random_state=RANDOM_STATE))])
#     p.fit(X_train, y_train)
#     ridge_coef_path.append(p.named_steps["model"].coef_)
# ridge_coef_path = np.array(ridge_coef_path)
# fig, ax = plt.subplots(figsize=(8, 5))
# for i in range(ridge_coef_path.shape[1]):
#     ax.plot(alphas, ridge_coef_path[:, i], linewidth=1, color=BAR_COLOR, alpha=0.5)
# ax.set_xscale("log")
# ax.set_title("Ridge Coefficient Path")
# plt.show()


### Challenge 2 — Target Transformation

`Freight`'ın dağılımı sağa çarpık (Section 1). `np.log1p` ile log transformation uygulayıp,
`TransformedTargetRegressor` kullanarak bu dönüşümün validation performansını nasıl etkilediğini
araştırın.


In [ ]:
# Advanced Challenge 2 — Target Transformation (optional):
# TODO: df["Freight"] içinde 0 veya negatif değer olup olmadığını KONTROL EDİN (log transform için gereklidir).
#       (Section 1'de bunu zaten doğruladık: min Freight = 0.02, yani > 0.)
# TODO: sklearn.compose.TransformedTargetRegressor kullanarak func=np.log1p, inverse_func=np.expm1 ile
#       bir LinearRegression pipeline'ı oluşturun.
# TODO: Validation MAE/RMSE/R2'yi, log-transform edilmemiş LinearRegression ile karşılaştırın.

# from sklearn.compose import TransformedTargetRegressor
# print("Min Freight:", df["Freight"].min())  # confirm strictly positive before using log1p
# log_target_pipe = Pipeline(steps=[
#     ("preprocessing", preprocessor),
#     ("model", TransformedTargetRegressor(regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1)),
# ])
# log_target_pipe.fit(X_train, y_train)
# pred_val_log = log_target_pipe.predict(X_val)
# print(mean_absolute_error(y_val, pred_val_log), r2_score(y_val, pred_val_log))


### Challenge 3 — Error Segmentation

Section 7'deki `ShipCountry` bazlı hata analizini genişletin: `ShipVia` (taşıyıcı) bazında da hata
segmentasyonu yapın ve iki segmentasyonu karşılaştırın.


In [ ]:
# Advanced Challenge 3 — Error Segmentation (optional):
# TODO: val_with_predictions içinde ShipVia bazında groupby yaparak MAE hesaplayın (Section 7'deki
#       ShipCountry analizine benzer şekilde).
# TODO: Küçük grupların (örn. n < 5) güvenilirliğinin sınırlı olduğunu açıkça belirtin.

# shipvia_mae = val_with_predictions.groupby("ShipVia").apply(
#     lambda d: pd.Series({"MAE": mean_absolute_error(d["Freight"], d["Predicted_Freight"]), "n": len(d)})
# )
# print(shipvia_mae)


---
## LAB05 Özeti

Bu LAB'da, LAB02-LAB04'te öğrendiğiniz bilgileri yeni bir dataset ve yeni bir problem (Freight
Charge Prediction) üzerinde kullanarak:

- Regression problemini tanımladık ve Classification'dan farkını açıkladık.
- Northwind sipariş verisinden, tahmin anında bilinmesi makul olan feature'ları seçtik (ve leakage
  riski taşıyan/yüksek kardinaliteli alanları gerekçeli biçimde dışarıda tuttuk).
- Veriyi kronolojik olarak train/validation/test kümelerine ayırdık.
- `DummyRegressor` ile baseline performansı belirledik.
- Simple Linear Regression'ın matematiğini bir synthetic örnekle, Multiple Linear Regression'ı ise
  gerçek Northwind verisiyle gösterdik.
- Ridge ve Lasso regularization'ı karşılaştırdık; `alpha`'nın katsayı küçülmesi/sıfırlanması üzerindeki
  etkisini somut sayılarla gözlemledik.
- Decision Tree Regression'da `max_depth`'in underfitting'den overfitting'e geçişi nasıl tetikledigini
  gördük.
- MAE, MSE, RMSE ve R²'yi hem matematiksel olarak hem Scikit-learn ile hesapladık ve her birinin farklı
  bir hikaye anlatabileceğini gördük.
- Residual analizi ile modelin hangi koşullarda daha az güvenilir olduğunu (heteroscedasticity, aşırı
  değerler) inceledik.
- Validation verisiyle bir model seçip, test verisini **yalnızca bir kez**, seçilen modelin bağımsız
  değerlendirmesi için kullandık.

Bu LAB'da eğittiğimiz preprocessing + model pipeline'ı, bir sonraki modülde genişletilebilir bir
temel oluşturur. Bir sonraki oturumda görüşmek üzere!
